In [1]:
import random
import time


__IMPORTANT__ = """
    khi obj.instance              se run __get__(self, instance, owner)
    khi obj.instance  = value      se run __set__(self, instance, value)
    khi del obj.instance           se run __delete__(self, instance)
    
    
    khi dung qua nhieu @property va .setter, nen dung Normal tranh lap lai code
"""


print(f"{'Normal':-^85}")
# Normal
class Normal(object):
    # def __init__(self, label):
    #     print("Normal.__init__")
    #     self.label = label

    #python 3.6 va moi hon
    #khi new des = Normal() => se run __set_name__ thay the cho __init__
    def __set_name__(self, ownercls, name):
        print("Normal.__set_name__")
        self.name = name

        """
            name = 'age'
            name = 'salary'
            name = 'rating'
            
            class Programmer:
               pass
               
            normal = Normal()
            Programmer.age = normal
            normal.__set_name__(Programmer, 'age')
            
            normal = Normal()
            Programmer.salary = normal
            normal.__set_name__(Programmer, 'salary')
            ...
            
        """
    
    # run khi <programmer.age> 
    def __get__(self, instance, ownercls):

        """
        self = Normal()
        instance = Programmer('Thong', 30, 5_000, 4)
        owner = Programmer
        """

        print("Normal.__get__(self, instance, ownercls):")

        return instance.__dict__[self.name] or 0
    
    # run khi <programmer.age = 38>
    def __set__(self, instance, value):
        print("Normal.__set__(self, instance, value):")
        instance.__dict__[self.name] = value

    # run khi <del programmer.age>
    def __delete__(self, instance):
        print("Normal.__delete__")

print("IMPORTANT scope global file")

class Programmer(object):
    print(
            "IMPORTANT trong class Programmer van run binh thuong  cung nhu la scope global file"
        )

    program = f"{'':*<10}class attribute"

    age = Normal()    #run __set_name__(self, owner, name)
    salary = Normal() #run __set_name__(self, owner, name)
    rating = Normal() #run __set_name__(self, owner, name)
    # khi class attrs = Normal trung ten instance attrs, thi se call class attr, default call instance attrs

    def __init__(self, age, salary, rating):
        self.age = age       #run Normal.__set__(self, instance, value):
        self.salary = salary #run Normal.__set__(self, instance, value):
        self.rating = rating #run Normal.__set__(self, instance, value):
        
        self.program = f"{'program':#^50} instance attribute "

print()

pro = Programmer(28, 5000, 10) #run Normal.__set__(self, instance, value): 3 lan
print()
print(f"{pro.age = }") #run Normal.__get__(self, instance, owner):

print()
print(f"{pro.salary = }")
print(f"{pro.program = }")

---------------------------------------Normal----------------------------------------
IMPORTANT scope global file
IMPORTANT trong class Programmer van run binh thuong  cung nhu la scope global file
Normal.__set_name__
Normal.__set_name__
Normal.__set_name__

Normal.__set__(self, instance, value):
Normal.__set__(self, instance, value):
Normal.__set__(self, instance, value):

Normal.__get__(self, instance, ownercls):
pro.age = 28

Normal.__get__(self, instance, ownercls):
pro.salary = 5000
pro.program = '#####################program###################### instance attribute '


In [2]:
print(f"{'LazyProperty':-^85}")

class LazyProperty:
    def __init__(self, function):
        print("LazyProperty.__init__")

        self.function = function
        self.funcname = function.__name__

    def __get__(self, instance, owner=None) -> object:
        print("LazyProperty.__get__")

        instance.__dict__[self.funcname] = self.function(instance)
        # deepthought.meaning_of_life = meaning_of_life(instance)=42
        return instance.__dict__[self.funcname]

    # def __set__(self, instance, value):
    #     pass

class DeepThought:

    @LazyProperty
    def meaning_of_life(self): # run LazyProperty.__get__
        time.sleep(3)
        return 42

    #meaning_of_life = LazyProperty(meaning_of_life)

    #deepthought.meaning_of_life lan dau se run run LazyProperty.__get__
    #deepthought.meaning_of_life lan sau chi lay value=42 tu __dict__
    #voi dieu kien ko co khai bao __set__
    
deepthought = DeepThought()
print(vars(deepthought))

print(deepthought.meaning_of_life)
print(deepthought.meaning_of_life)
print(deepthought.meaning_of_life)

print(vars(deepthought))

------------------------------------LazyProperty-------------------------------------
LazyProperty.__init__
{}
LazyProperty.__get__
42
42
42
{'meaning_of_life': 42}


In [3]:
print(f"{'EvenNumber':-^85}")

class EvenNumber:
    def __set_name__(self, owner, name):
        print("EvenNumber.__set_name__")
        print(f"owner: {owner}\nname: {name}")
        self.name = name

    def __get__(self, instance, owner=None):
        print("EvenNumber.__get__")
        return instance.__dict__.get(self.name) or 0

    def __set__(self, instance, value):
        print("EvenNumer.__set__")
        instance.__dict__[self.name] = (value if value % 2 == 0 else 0)

class Values:
    value1 = EvenNumber() #run EvenNumber.__set_name__
    value2 = EvenNumber() #run EvenNumber.__set_name__
    value3 = EvenNumber() #run EvenNumber.__set_name__
    

my_values = Values() 
print()
my_values.value1 = 1
my_values.value2 = 4
print()
print(f"{my_values.value1 = }")
print(f"{my_values.value2 = }")

-------------------------------------EvenNumber--------------------------------------
EvenNumber.__set_name__
owner: <class '__main__.Values'>
name: value1
EvenNumber.__set_name__
owner: <class '__main__.Values'>
name: value2
EvenNumber.__set_name__
owner: <class '__main__.Values'>
name: value3

EvenNumer.__set__
EvenNumer.__set__

EvenNumber.__get__
my_values.value1 = 0
EvenNumber.__get__
my_values.value2 = 4


In [4]:
class Property(object):
    "Emulate PyProperty_Type() in Objects/descrobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__) #new Property()

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__) #new Property()

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__) #new Property()

In [5]:
import types

class Function(object):

    def __get__(self, obj, objtype=None):
        "Simulate func_descr_get() in Objects/funcobject.c"
        
        if obj is None:
            print("Tao la function")
            return self

        print('Tao la method')
        return types.MethodType(self, obj)
        # types.MethodType(self, instance) return self

    def __call__(self, *args, **kwds):
        pass

    
class Human:
    name = 'dung'
    old = 28
    
    def get_name(self):
        return self.name, self.old
        

print(Human.get_name)
print()
print(Human().get_name())

<function Human.get_name at 0x7f496c1c33a0>

('dung', 28)


In [6]:
class StaticMethod(object):
    "Emulate PyStaticMethod_Type() in Objects/funcobject.c"

    def __init__(self, f):
        self.f = f

    def __get__(self, obj, objtype=None):
        return self.f

In [7]:
class ClassMethod(object):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, f):
        self.f = f

    def __get__(self, obj, klass=None):
        if klass is None:
            klass = type(obj)
        def newfunc(*args):
            return self.f(klass, *args)
        return newfunc